# SEN.AI — Gemma 4 E4B Colab QLoRA Eğitimi

Bu defter, `senai_aureon_e4b_dataset.jsonl` dosyasındaki gözden geçirilmiş çok turlu sohbetleri Gemma 4 E4B ile eğitir. Veri kümesini yeniden üretmek için önce `python colab/build_aureon_dataset.py` çalıştırın.

- QLoRA / rsLoRA; asistan yanıtlarına loss uygulanır, kullanıcı ve system metinleri hedef olarak eğitilmez.
- Kimlik yalnızca SEN.AI; `dev_plan`/`patch`, kart biçimleri, kaynaklı araştırma, arama izni ve kaynak seçimi örnekleri içerir.
- GPU belleğine göre bağlam uzunluğu 2K, 4K veya 8K seçilir. T4 üzerinde OOM olursa daha güçlü GPU kullanın.
- GGUF dışa aktarımı ayrıca denenir; dönüştürme başarısız olsa bile PEFT adaptör ZIP'i indirilir.

- Verinin %15’i doğrulama için ayrılır; eğitim öncesi başlangıç ölçümü ve her epoch sonunda eval_loss alınır, kayıp grafiği ve en iyi checkpoint gösterilir.

- Kart şeması: `dev_plan` ve `media_list` her zaman `items` dizisi kullanır; `steps` yalnızca `checklist` kartına aittir. Hızlı test hücresi kart JSON’unu otomatik doğrular.


Kart eğitimi için eklenen hedefli örnekler train bölümüne sabitlenir; doğrulama ve bağımsız test örnekleri ayrı tutulur.
- Split policy: prompt-pinned examples are assigned explicitly to train, validation, or test; the three prompt lists are disjoint and missing examples stop the run.


In [ ]:
# 1. Uyumlu kütüphaneler
%%capture
!pip install -q "transformers==5.5.0" "trl" "datasets>=3.4.1" sentencepiece protobuf huggingface_hub hf_transfer
!pip install -q --no-deps bitsandbytes accelerate xformers peft triton cut_cross_entropy unsloth_zoo
!pip install -q --no-deps unsloth


In [ ]:
# 2. GPU belleğine göre E4B QLoRA yapılandırması
from unsloth import FastLanguageModel
import torch

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
vram_gb = (torch.cuda.get_device_properties(0).total_memory / 1e9) if torch.cuda.is_available() else 0
if not torch.cuda.is_available():
    raise RuntimeError("Eğitim için CUDA GPU gerekli.")

USE_BF16 = torch.cuda.is_bf16_supported()
MAX_SEQ_LENGTH = 8192 if vram_gb >= 36 else 4096 if vram_gb >= 20 else 2048
BATCH_SIZE = 2 if vram_gb >= 36 else 1
GRAD_ACCUM = 4 if BATCH_SIZE == 2 else 8
print(f"GPU: {gpu_name} ({vram_gb:.1f} GB) | bf16={USE_BF16} | ctx={MAX_SEQ_LENGTH} | effective batch={BATCH_SIZE * GRAD_ACCUM}")

MODEL_NAME = "unsloth/gemma-4-E4B-it-unsloth-bnb-4bit"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    full_finetuning=False,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=128,
    lora_dropout=0.0,
    bias="none",
    use_rslora=True,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)


In [ ]:
# 3. Veriyi yükle ve kaynak gruplarını koruyarak üç kümeye ayır
from google.colab import files
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
from collections import defaultdict, Counter
import numpy as np
import re

tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
uploaded = files.upload()  # senai_aureon_e4b_dataset.jsonl dosyasını seçin
if not uploaded:
    raise RuntimeError("JSONL veri dosyası seçilmedi.")
dataset_path = next(iter(uploaded))
raw_ds = load_dataset("json", data_files=dataset_path, split="train")
if len(raw_ds) < 30 or "messages" not in raw_ds.column_names:
    raise ValueError("Üçlü bölme için en az 30 kayıt ve messages alanı gerekli.")

def task_category(example):
    messages = example["messages"]
    user_text = " ".join(m.get("content", "") for m in messages if m.get("role") == "user").casefold()
    assistant_text = " ".join(m.get("content", "") for m in messages if m.get("role") == "assistant")
    if any(x in user_text for x in ("sen kimsin", "seni kim yaptı", "adın ne", "hangi yapay zekâ modelisin")):
        return "identity"
    if "```sen_card" in assistant_text or any(x in user_text for x in ("kart", "görsel", "grafik", "tablo", "checklist")):
        return "structured_output"
    if sum(m.get("role") == "user" for m in messages) > 1:
        return "multi_turn"
    if any(x in user_text for x in ("2025", "2026", "güncel", "kaynak", "türkiye", "dünya gündem")):
        return "current_or_factual"
    if any(x in user_text for x in ("kod", "yazılım", "python", "html", "css", "javascript", "fonksiyon")):
        return "coding"
    return "general"

categories = [task_category(row) for row in raw_ds]
category_names = sorted(set(categories))
# Kume etiketleri JSONL kayitlarindan okunur; etiketsiz eski kayitlar seed ile ayrilir.
partition_indices = {"train": set(), "validation": set(), "test": set()}
for index, row in enumerate(raw_ds):
    partition = row.get("partition")
    if partition in partition_indices:
        partition_indices[partition].add(index)
required_counts = {"train": 88, "validation": 21, "test": 22}
for name, expected in required_counts.items():
    if len(partition_indices[name]) != expected:
        raise RuntimeError(f"{name} sabit kayit sayisi beklenen {expected}; bulunan {len(partition_indices[name])}. Guncel 450 kayitli JSONL dosyasini yukleyin.")

# URL kaynak grupları aynı partition'a sabitlenir; çakışma veri sızıntısı sayılıp durdurulur.
for group, indices in source_groups.items():
    labels = {raw_ds[i].get("partition") for i in indices if raw_ds[i].get("partition") in ("train", "validation", "test")}
    if len(labels) > 1:
        raise RuntimeError(f"Ayni kaynak grubu birden fazla partition'da: {group}")

rng = np.random.default_rng(3407)
train_indices, eval_indices, test_indices = [], [], []
for category in category_names:
    free_units = []
    for unit in units_by_category[category]:
        labels = {raw_ds[i].get("partition") for i in unit if raw_ds[i].get("partition") in ("train", "validation", "test")}
        if labels:
            label = next(iter(labels))
            target = train_indices if label == "train" else eval_indices if label == "validation" else test_indices
            target.extend(unit)
        else:
            free_units.append(unit)
    order = rng.permutation(len(free_units))
    free_units = [free_units[i] for i in order]
    if len(free_units) >= 5:
        n_test = max(1, round(len(free_units) * 0.15))
        n_eval = max(1, round((len(free_units) - n_test) * 0.15))
    elif len(free_units) >= 3:
        n_test, n_eval = 1, 1
    elif len(free_units) == 2:
        n_test, n_eval = 0, 1
    else:
        n_test, n_eval = 0, 0
    for unit in free_units[:n_test]: test_indices.extend(unit)
    for unit in free_units[n_test:n_test + n_eval]: eval_indices.extend(unit)
    for unit in free_units[n_test + n_eval:]: train_indices.extend(unit)

train_indices, eval_indices, test_indices = sorted(set(train_indices)), sorted(set(eval_indices)), sorted(set(test_indices))
if set(train_indices) & set(eval_indices) or set(train_indices) & set(test_indices) or set(eval_indices) & set(test_indices):
    raise RuntimeError("Train/validation/test index sets overlap.")
train_raw = raw_ds.select(train_indices)
eval_raw = raw_ds.select(eval_indices)
test_raw = raw_ds.select(test_indices)
if not len(eval_raw) or not len(test_raw):
    raise RuntimeError("Validation or test partition is empty; inspect category split.")

def format_row(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False).removeprefix("<bos>")
    return {"text": text}

train_ds = train_raw.map(format_row, remove_columns=train_raw.column_names)
eval_ds = eval_raw.map(format_row, remove_columns=eval_raw.column_names)
test_ds = test_raw.map(format_row, remove_columns=test_raw.column_names)
print(f"Toplam: {len(raw_ds)} | Eğitim: {len(train_ds)} | Doğrulama: {len(eval_ds)} | Ayrı test: {len(test_ds)}")
print("Görev türlerine göre kayıt sayıları:")
for name in category_names:
    print(f"  {name}: toplam={categories.count(name)}")
print("Seed=3407. Aynı kaynak URL'sinden türetilen kayıtlar aynı kümede kalır; test checkpoint seçimine katılmaz.")


In [ ]:
# 4. QLoRA eğitimi, başlangıç ölçümü ve ayrıntılı değerlendirme grafiği
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_ratio=0.08,
        num_train_epochs=3,
        learning_rate=1.2e-4,
        fp16=not USE_BF16,
        bf16=USE_BF16,
        logging_steps=1,
        eval_strategy="epoch",
        eval_on_start=False,
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        optim="adamw_8bit",
        weight_decay=0.02,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|turn>user\n",
    response_part="<|turn>model\n",
)

# Eğitilmemiş temel modeli aynı doğrulama kümesinde ayrı ölç.
print("Eğitim öncesi temel model ölçümü alınıyor...")
baseline_metrics = trainer.evaluate(metric_key_prefix="eval_before_training")
baseline_loss = baseline_metrics.get("eval_before_training_loss")
if baseline_loss is None:
    raise RuntimeError(f"Eğitim öncesi eval loss alınamadı: {baseline_metrics}")
print(f"Eğitim öncesi doğrulama loss: {baseline_loss:.4f}")

trainer_stats = trainer.train()
print("Eğitim tamamlandı.", trainer_stats)
print("En iyi doğrulama checkpoint'i:", trainer.state.best_model_checkpoint)
# Test kümesi eğitimde/checkpoint seçiminde kullanılmadı; şimdi bir kez ölçülür.
test_metrics = trainer.evaluate(eval_dataset=test_ds, metric_key_prefix="test")
test_loss = test_metrics.get("test_loss")
if test_loss is None:
    raise RuntimeError(f"Ayrı test loss alınamadı: {test_metrics}")
print(f"Ayrı test loss (en iyi doğrulama checkpoint'i): {test_loss:.4f}")

import math
import pandas as pd
import matplotlib.pyplot as plt
history = pd.DataFrame(trainer.state.log_history)
if "epoch" not in history or "eval_loss" not in history:
    raise RuntimeError("Eğitim geçmişinde epoch/eval_loss bulunamadı.")
eval_logs = history.dropna(subset=["eval_loss", "epoch"])[["epoch", "eval_loss"]].copy()
eval_logs["epoch"] = eval_logs["epoch"].round(2)
eval_by_epoch = eval_logs.groupby("epoch", as_index=False).last()
train_logs = history.dropna(subset=["loss", "epoch"])[["epoch", "loss"]].copy()
train_logs["epoch"] = train_logs["epoch"].apply(lambda value: max(1, math.ceil(value)))
train_by_epoch = train_logs.groupby("epoch", as_index=False).last().rename(columns={"loss": "train_loss"})
post_training = eval_by_epoch.merge(train_by_epoch, on="epoch", how="left")
if post_training.empty:
    raise RuntimeError("Epoch sonu doğrulama ölçümü bulunamadı.")
post_training["eval_perplexity"] = post_training["eval_loss"].clip(upper=20).apply(math.exp)
best_row = post_training.loc[post_training["eval_loss"].idxmin()]
print(f"En düşük doğrulama loss: {best_row['eval_loss']:.4f} | epoch: {int(best_row['epoch'])} | perplexity: {best_row['eval_perplexity']:.2f}")
print(f"En iyi checkpoint üzerindeki ayrı test loss: {test_loss:.4f}")
if "grad_norm" in history:
    grad_rows = history.dropna(subset=["grad_norm"])["grad_norm"]
    if not grad_rows.empty: print(f"Gradyan normu: medyan={grad_rows.median():.4f}, en yüksek={grad_rows.max():.4f}")
if "learning_rate" in history:
    lr_rows = history.dropna(subset=["learning_rate"])["learning_rate"]
    if not lr_rows.empty: print(f"Öğrenme oranı: başlangıç={lr_rows.iloc[0]:.2e}, son={lr_rows.iloc[-1]:.2e}")

metrics = post_training.copy()
metrics.insert(0, "ölçüm", "Epoch")
metrics = pd.concat([
    pd.DataFrame([{"ölçüm": "Eğitim öncesi temel model", "epoch": 0.0, "eval_loss": baseline_loss, "train_loss": float("nan"), "eval_perplexity": math.exp(min(baseline_loss, 20))}]),
    metrics,
], ignore_index=True)
display(metrics.round(4))

# Ölçekleri ayrı panellerde göster; test yalnızca en iyi checkpoint'te tek ölçüm olarak işaretlenir.
fig, axes = plt.subplots(3, 1, figsize=(10, 11), dpi=120)
ax = axes[0]
ax.plot(metrics["epoch"], metrics["eval_loss"], color="#2457A7", marker="o", linewidth=2.4, label="Doğrulama loss")
ax.plot(metrics["epoch"], metrics["train_loss"], color="#D97706", marker="s", linewidth=2, label="Eğitim loss (epoch sonu)")
ax.scatter([0], [baseline_loss], color="#6B7280", marker="D", s=85, zorder=5, label="Eğitim öncesi temel model")
ax.scatter([best_row["epoch"]], [best_row["eval_loss"]], color="#15803D", edgecolor="white", s=145, zorder=6, label=f"En iyi epoch: {int(best_row['epoch'])}")
ax.scatter([best_row["epoch"]], [test_loss], color="#BE185D", marker="P", edgecolor="white", s=130, zorder=6, label="Ayrı test (tek ölçüm)")
ax.set_title("Model kalitesi: eğitim, doğrulama ve ayrı test")
ax.set_ylabel("Loss (düşük olması genelde daha iyidir)")
ax.set_xticks(sorted(set([0] + [int(e) for e in post_training["epoch"]])))
ax.grid(True, linestyle="--", alpha=0.3)
ax.legend(frameon=False, ncol=2)
ax2 = ax.twinx()
ax2.plot(post_training["epoch"], post_training["eval_perplexity"], color="#7C3AED", marker="^", linestyle="--", label="Doğrulama perplexity")
ax2.set_ylabel("Perplexity (düşük olması genelde daha iyidir)", color="#7C3AED")
ax2.tick_params(axis="y", labelcolor="#7C3AED")

if "learning_rate" in history and not history.dropna(subset=["learning_rate", "epoch"]).empty:
    lr = history.dropna(subset=["learning_rate", "epoch"])
    axes[1].plot(lr["epoch"], lr["learning_rate"], color="#0891B2", linewidth=2)
    axes[1].set_title("Öğrenme oranı")
    axes[1].set_ylabel("Learning rate")
    axes[1].ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    axes[1].grid(True, linestyle="--", alpha=0.3)
else:
    axes[1].text(0.5, 0.5, "Loglarda learning_rate ölçümü yok", ha="center", va="center")

if "grad_norm" in history and not history.dropna(subset=["grad_norm", "epoch"]).empty:
    grad = history.dropna(subset=["grad_norm", "epoch"])
    axes[2].plot(grad["epoch"], grad["grad_norm"], color="#DC2626", alpha=0.75, linewidth=1.4, label="Gradyan normu")
    axes[2].axhline(grad["grad_norm"].median(), color="#7F1D1D", linestyle="--", label=f"Medyan: {grad['grad_norm'].median():.3f}")
    axes[2].set_title("Gradyan normu — optimizasyon tanısı")
    axes[2].set_ylabel("grad_norm")
    axes[2].legend(frameon=False)
    axes[2].grid(True, linestyle="--", alpha=0.3)
else:
    axes[2].text(0.5, 0.5, "Loglarda grad_norm ölçümü yok", ha="center", va="center")
axes[2].set_xlabel("Epoch (0 = eğitim öncesi)")
fig.suptitle("Eğitim süreci: kalite ve optimizasyon göstergeleri", fontsize=15, y=1.01)
fig.tight_layout()
plt.show()


In [ ]:
# 5. Genişletilmiş hızlı çıktı ve görev testi
import json
import re
from collections import Counter

FastLanguageModel.for_inference(model)
SYS_CORE = raw_ds[0]["messages"][0]["content"]

def text_msg(role, text):
    return {"role": role, "content": [{"type": "text", "text": text}]}

def run_test(label, messages, max_new_tokens=700):
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to("cuda")
    input_len = inputs["input_ids"].shape[-1]
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
    answer = tokenizer.decode(output[0][input_len:], skip_special_tokens=True)
    print(f"=== {label} ===")
    print(answer)
    print()
    return answer

def validate_card(label, answer, expected_kind, required_fields, min_items=None, item_fields=()):
    blocks = re.findall(r"```sen_card\s*\n(.*?)\n```", answer, re.S)
    if len(blocks) != 1:
        print(f"[KART] {label}: beklenen 1 kart, bulunan {len(blocks)}")
        return False
    try:
        card = json.loads(blocks[0])
    except json.JSONDecodeError as exc:
        print(f"[KART] {label}: JSON hatası: {exc}")
        return False
    missing = [field for field in required_fields if field not in card]
    ok = card.get("kind") == expected_kind and not missing
    print(f"[KART] {label}: kind={card.get('kind')} | eksik={missing or 'yok'}")
    if card.get("kind") == "dev_plan":
        if card.get("mode") == "select": ok &= isinstance(card.get("items"), list) and len(card["items"]) >= 2
        elif card.get("mode") == "confirm": ok &= isinstance(card.get("selected"), list)
        else: ok = False
    if min_items is not None:
        collection_key = "steps" if expected_kind == "checklist" else "items"
        items = card.get(collection_key)
        if not isinstance(items, list) or len(items) < min_items:
            print(f"[KART] {label}: {collection_key} sayısı en az {min_items} olmalı")
            ok = False
        elif item_fields:
            for ix, item in enumerate(items):
                absent = [field for field in item_fields if not item.get(field)] if isinstance(item, dict) else list(item_fields)
                if absent:
                    print(f"[KART] {label}: items[{ix}] eksik alan={absent}")
                    ok = False
            if expected_kind == "media_list":
                titles = [item.get("title", "").strip().casefold() for item in items if isinstance(item, dict)]
                if len(titles) != len(set(titles)):
                    print(f"[KART] {label}: tekrar eden başlık var")
                    ok = False
    print(f"[KART] {label}: {'GEÇTİ' if ok else 'BAŞARISIZ'}")
    return bool(ok)

results = {}
def record(name, passed):
    results[name] = bool(passed)

# 1–2: Kimlik, model/sağlayıcı adlarını SEN.AI kimliği gibi sunmamalı.
identity_1 = run_test("Kimlik: model sorusu", [text_msg("system", SYS_CORE), text_msg("user", "Gemma 4 müsün, seni Google mı yaptı?")], 180)
record("Kimlik yanıtı", "SEN.AI" in identity_1 and not any(x in identity_1.casefold() for x in ("aureon 1.1", "sage 1", "orion", "gemma", "google")))
identity_2 = run_test("Kimlik: adın", [text_msg("system", SYS_CORE), text_msg("user", "Adın ne, kısaca anlatır mısın?")], 150)
record("Kısa kimlik tutarlılığı", "SEN.AI" in identity_2)

# 3–4: Tek mesajlı kart istekleri.
coffee = run_test("Kahve sitesi planı", [text_msg("system", SYS_CORE), text_msg("user", "Modern bir kahve evi sitesi yapmak istiyorum, ne önerirsin?")])
record("dev_plan şeması", validate_card("Kahve sitesi", coffee, "dev_plan", ["title", "mode", "items"], min_items=2))
series = run_test("Beş Türk dizisi kartı", [text_msg("system", SYS_CORE), text_msg("user", "Farklı türlerden beş Türk dizisini türleri ve kısa açıklamalarıyla görsel kart formatında sun.")], 900)
record("media_list şeması/çeşitliliği", validate_card("Dizi önerileri", series, "media_list", ["title", "items"], min_items=5, item_fields=("title", "subtitle", "image_query")))

# 5: Çok turlu bağlam ve kart devam isteği.
series_multi = run_test("Çok turlu dizi kartı", [
    text_msg("system", SYS_CORE),
    text_msg("user", "Farklı türlerden beş Türk dizisi öner; sıralamanın öznel olduğunu belirt."),
    text_msg("assistant", "Kişisel önerilerim: Kurtlar Vadisi (aksiyon/suç), Ezel (dram/suç), Behzat Ç. (polis/dram), Avrupa Yakası (komedi) ve Muhteşem Yüzyıl (tarihî dram)."),
    text_msg("user", "Bunları görsel kart şeklinde sun."),
], 900)
record("Çok turlu media_list", validate_card("Bağlamı koruyan dizi kartı", series_multi, "media_list", ["title", "items"], min_items=5, item_fields=("title", "subtitle", "image_query")))

# 6: Checklist kartı.
checklist = run_test("Haftalık kontrol listesi", [text_msg("system", SYS_CORE), text_msg("user", "Haftalık verimli çalışma kontrol listesi oluştur.")], 500)
record("checklist şeması", validate_card("Kontrol listesi", checklist, "checklist", ["title", "steps"], min_items=3, item_fields=("label",)))

# 7: Kaynak gerektiren bilgi; kaynak verilmeden araştırma yaptığını iddia etmemeli.
current = run_test("Kaynak ve güncellik", [text_msg("system", SYS_CORE), text_msg("user", "Bugünkü Türkiye işsizlik oranı kaç? Kaynak göstermeden kesin sayı verme.")], 250)
record("Güncel bilgide belirsizlik", any(x in current.casefold() for x in ("güncel", "kaynak", "doğrula", "kontrol")))

# 8: Tarihli eğitim bilgisini kaynağıyla yanıtlayabilme.
known = run_test("Tarihli kaynaklı bilgi", [text_msg("system", SYS_CORE), text_msg("user", "TÜİK'e göre 2025 Türkiye nüfusu kaçtı? Tarihi ve kaynağı belirt.")], 350)
record("Kaynaklı nüfus yanıtı", "86.092.168" in known.replace(" ", "") and "tuik" in known.casefold())

# 9: Çok adımlı değişiklikte patch formatı.
patch = run_test("Değişiklik isteği ve patch", [
    text_msg("system", SYS_CORE),
    text_msg("user", "Mevcut başlığı ‘Hoş geldin’ metni ve rengi #385f4a olacak şekilde değiştirmek istiyorum. Önce ne yapacağını söyle."),
], 450)
record("Kod öncesi dev_plan", validate_card("Kod öncesi plan", patch, "dev_plan", ["title", "mode", "items"], min_items=2))

# 10: Grafik kartında kullanıcı tarafından verilen değerlerin korunması.
chart = run_test("Grafik verisi", [text_msg("system", SYS_CORE), text_msg("user", "Bu veriyi grafikle göster: Ocak 12, Şubat 16, Mart 14. Değerleri değiştirme.")], 500)
def validate_chart(label, answer):
    blocks = re.findall(r"```chart\s*\n(.*?)\n```", answer, re.S)
    if len(blocks) != 1:
        print(f"[GRAFİK] {label}: beklenen 1 chart bloğu, bulunan {len(blocks)}")
        return False
    try: data = json.loads(blocks[0])
    except json.JSONDecodeError as exc:
        print(f"[GRAFİK] {label}: JSON hatası: {exc}")
        return False
    ok = data.get("kind") == "chart" and data.get("labels") == ["Ocak", "Şubat", "Mart"] and data.get("values") == [12, 16, 14]
    print(f"[GRAFİK] {label}: {'GEÇTİ' if ok else 'BAŞARISIZ — etiket/değer/kind uyuşmuyor'}")
    return ok
record("Grafik JSON ve değerleri koruma", validate_chart("Aylık değerler", chart))

print("\n=== TEST ÖZETİ ===")
for name, passed in results.items(): print(f"{'GEÇTİ' if passed else 'BAŞARISIZ'} — {name}")
print(f"Toplam: {sum(results.values())}/{len(results)} geçti")


In [ ]:
# 6. PEFT adaptörünü indir; GGUF dönüşümünü ayrıca dene
import os
import shutil
import subprocess
from google.colab import files

SAVE_DIR = "senai_aureon_e4b_lora"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
shutil.make_archive("senai_aureon_e4b_lora", "zip", SAVE_DIR)
files.download("senai_aureon_e4b_lora.zip")

# llama.cpp GGUF LoRA dönüştürücüsü Gemma 4 yapılandırmalarında sürüme
# bağlı sorun çıkarabilir. ZIP önceden indirildiği için eğitim çıktısı korunur.
try:
    if not os.path.isdir("/content/llama_cpp_conv"):
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp.git", "/content/llama_cpp_conv"],
            check=True,
        )
    subprocess.run(["pip", "install", "-q", "gguf"], check=True)
    subprocess.run(
        [
            "python", "/content/llama_cpp_conv/convert_lora_to_gguf.py",
            "--base-model-id", MODEL_NAME,
            "--outfile", "senai_aureon_e4b_lora.gguf",
            "--outtype", "f16",
            SAVE_DIR,
        ],
        check=True,
    )
    files.download("senai_aureon_e4b_lora.gguf")
    print("GGUF adaptörünü yerel SEN.AI kurulumunda models/loras/aureon_v1/ altına koyabilirsiniz.")
except (subprocess.CalledProcessError, FileNotFoundError) as exc:
    print(f"GGUF dönüşümü tamamlanamadı ({exc}). PEFT adaptör ZIP'i zaten indirildi.")
